# Run reliability experiment on SAKT

## Import dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/drive/MyDrive/education-ml-research/ASSISTments2009"

ORIGINAL_DATA = os.path.join(
    DATA_DIR,
    "skill_builder_data.csv"
)

QUARTILE_DATA = os.path.join(
    DATA_DIR,
    "student_ability_quartiles.csv"
)

OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "reliability_experiment"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Original dataset:", ORIGINAL_DATA)
print("Quartile dataset:", QUARTILE_DATA)
print("Output directory:", OUTPUT_DIR)

df = pd.read_csv(ORIGINAL_DATA, encoding="latin1")

ability_df = pd.read_csv(QUARTILE_DATA, encoding="latin1")

print("Original dataset shape:", df.shape)
print("Ability dataset shape:", ability_df.shape)

print("\nAbility quartile counts:")
print(
    ability_df["ability_quartile"]
    .value_counts()
    .sort_index()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original dataset: /content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv
Quartile dataset: /content/drive/MyDrive/education-ml-research/ASSISTments2009/student_ability_quartiles.csv
Output directory: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment
Original dataset shape: (525534, 30)
Ability dataset shape: (4217, 5)

Ability quartile counts:
ability_quartile
Q1    1056
Q2    1053
Q3    1056
Q4    1052
Name: count, dtype: int64


/tmp/ipykernel_4962/2504488265.py:31: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ORIGINAL_DATA, encoding="latin1")


## Create 80/20 split within each quartile

In [ ]:
RANDOM_SEED = 42
TRAIN_RATIO = 0.80

rng = np.random.default_rng(RANDOM_SEED)

train_students = []
test_students_by_quartile = {}

print("Creating stratified student split...\n")

for quartile in ["Q1", "Q2", "Q3", "Q4"]:

    # Get students belonging to this quartile
    students = (
        ability_df[
            ability_df["ability_quartile"] == quartile
        ]["user_id"]
        .unique()
    )

    # Shuffle students
    students = students.copy()
    rng.shuffle(students)

    # 80/20 split
    split_idx = int(len(students) * TRAIN_RATIO)

    q_train_students = students[:split_idx]
    q_test_students = students[split_idx:]

    # Add training students to the global training set
    train_students.extend(q_train_students)

    # Save test students separately by quartile
    test_students_by_quartile[quartile] = q_test_students

    print(
        f"{quartile}: "
        f"{len(students)} total | "
        f"{len(q_train_students)} train | "
        f"{len(q_test_students)} test"
    )

train_students = np.array(train_students)

print("\nTotal training students:", len(train_students))
print(
    "Total test students:",
    sum(len(x) for x in test_students_by_quartile.values())
)

Creating stratified student split...

Q1: 1056 total | 844 train | 212 test
Q2: 1053 total | 842 train | 211 test
Q3: 1056 total | 844 train | 212 test
Q4: 1052 total | 841 train | 211 test

Total training students: 3371
Total test students: 846


## Verify there is no overlap

In [ ]:
train_student_set = set(train_students)

all_test_students = set(
    np.concatenate(
        list(test_students_by_quartile.values())
    )
)

overlap = train_student_set & all_test_students

print("Training students:", len(train_student_set))
print("Test students:", len(all_test_students))
print("Overlap:", len(overlap))

assert len(overlap) == 0, "ERROR: train/test students overlap!"

print("✓ No student appears in both training and testing.")

Training students: 3371
Test students: 846
Overlap: 0
✓ No student appears in both training and testing.


## Create interaction datasets

In [ ]:
# Training interactions
train_df = df[
    df["user_id"].isin(train_students)
].copy()

# Test interactions for each quartile
q1_test_df = df[
    df["user_id"].isin(test_students_by_quartile["Q1"])
].copy()

q2_test_df = df[
    df["user_id"].isin(test_students_by_quartile["Q2"])
].copy()

q3_test_df = df[
    df["user_id"].isin(test_students_by_quartile["Q3"])
].copy()

q4_test_df = df[
    df["user_id"].isin(test_students_by_quartile["Q4"])
].copy()

print("Training interactions:", len(train_df))
print("Q1 test interactions:", len(q1_test_df))
print("Q2 test interactions:", len(q2_test_df))
print("Q3 test interactions:", len(q3_test_df))
print("Q4 test interactions:", len(q4_test_df))

Training interactions: 426580
Q1 test interactions: 11444
Q2 test interactions: 23524
Q3 test interactions: 30184
Q4 test interactions: 33802


## Save the 5 datasets

In [ ]:
train_path = os.path.join(
    OUTPUT_DIR,
    "train.csv"
)

q1_path = os.path.join(
    OUTPUT_DIR,
    "q1_test.csv"
)

q2_path = os.path.join(
    OUTPUT_DIR,
    "q2_test.csv"
)

q3_path = os.path.join(
    OUTPUT_DIR,
    "q3_test.csv"
)

q4_path = os.path.join(
    OUTPUT_DIR,
    "q4_test.csv"
)

train_df.to_csv(train_path, index=False)
q1_test_df.to_csv(q1_path, index=False)
q2_test_df.to_csv(q2_path, index=False)
q3_test_df.to_csv(q3_path, index=False)
q4_test_df.to_csv(q4_path, index=False)

print("Saved:")
print(train_path)
print(q1_path)
print(q2_path)
print(q3_path)
print(q4_path)

Saved:
/content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/train.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/q1_test.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/q2_test.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/q3_test.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/q4_test.csv


## Final verification

In [ ]:
print("========== FINAL CHECK ==========")

datasets = {
    "Train": train_df,
    "Q1 Test": q1_test_df,
    "Q2 Test": q2_test_df,
    "Q3 Test": q3_test_df,
    "Q4 Test": q4_test_df
}

for name, data in datasets.items():

    print(
        f"{name}: "
        f"{len(data):,} interactions | "
        f"{data['user_id'].nunique():,} students"
    )

# Check all test groups are mutually exclusive
q1_students = set(q1_test_df["user_id"])
q2_students = set(q2_test_df["user_id"])
q3_students = set(q3_test_df["user_id"])
q4_students = set(q4_test_df["user_id"])

assert not (q1_students & q2_students)
assert not (q1_students & q3_students)
assert not (q1_students & q4_students)
assert not (q2_students & q3_students)
assert not (q2_students & q4_students)
assert not (q3_students & q4_students)

print("\n✓ Q1-Q4 test students are mutually exclusive.")

# Check train/test separation
assert not (train_student_set & q1_students)
assert not (train_student_set & q2_students)
assert not (train_student_set & q3_students)
assert not (train_student_set & q4_students)

print("✓ Training students do not appear in any test set.")

print("\nAll checks passed.")

========== FINAL CHECK ==========
Train: 426,580 interactions | 3,371 students
Q1 Test: 11,444 interactions | 212 students
Q2 Test: 23,524 interactions | 211 students
Q3 Test: 30,184 interactions | 212 students
Q4 Test: 33,802 interactions | 211 students

✓ Q1-Q4 test students are mutually exclusive.
✓ Training students do not appear in any test set.

All checks passed.


## Clone github repo

In [ ]:
!git clone https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git
%cd /content/knowledge-tracing-collection-pytorch
!git remote -v

Cloning into 'knowledge-tracing-collection-pytorch'...
remote: Enumerating objects: 488, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 488 (delta 22), reused 4 (delta 0), pack-reused 441 (from 1)
Receiving objects: 100% (488/488), 1.11 MiB | 20.35 MiB/s, done.
Resolving deltas: 100% (252/252), done.
/content/knowledge-tracing-collection-pytorch
origin	https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git (fetch)
origin	https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git (push)


## Imports and random seed for reproducibility

In [ ]:
import os
import random
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.metrics import roc_auc_score

from models.sakt import SAKT
from models.utils import match_seq_len, collate_fn

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())

Seed: 42
CUDA available: True


## Import and check datasets

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RELIABILITY_DIR = (
    "/content/drive/MyDrive/education-ml-research/"
    "ASSISTments2009/reliability_experiment"
)

TRAIN_PATH = os.path.join(
    RELIABILITY_DIR,
    "train.csv"
)

TEST_PATHS = {
    "Q1": os.path.join(RELIABILITY_DIR, "q1_test.csv"),
    "Q2": os.path.join(RELIABILITY_DIR, "q2_test.csv"),
    "Q3": os.path.join(RELIABILITY_DIR, "q3_test.csv"),
    "Q4": os.path.join(RELIABILITY_DIR, "q4_test.csv"),
}

print("Training file exists:", os.path.exists(TRAIN_PATH))

for q, path in TEST_PATHS.items():
    print(q, "exists:", os.path.exists(path))

train_df = pd.read_csv(TRAIN_PATH)

test_dfs = {
    q: pd.read_csv(path)
    for q, path in TEST_PATHS.items()
}

print("Training shape:", train_df.shape)

for q, df_q in test_dfs.items():
    print(f"{q} shape:", df_q.shape)

print(
    "Training students:",
    train_df["user_id"].nunique()
)

print("Testing students:")
for q, df_q in test_dfs.items():
    print(
        f"{q} students:",
        df_q["user_id"].nunique()
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training file exists: True
Q1 exists: True
Q2 exists: True
Q3 exists: True
Q4 exists: True


/tmp/ipykernel_5197/1181723478.py:27: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_PATH)


Training shape: (426580, 30)
Q1 shape: (11444, 30)
Q2 shape: (23524, 30)
Q3 shape: (30184, 30)
Q4 shape: (33802, 30)
Training students: 3371
Testing students:
Q1 students: 212
Q2 students: 211
Q3 students: 212
Q4 students: 211


/tmp/ipykernel_5197/1181723478.py:30: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  q: pd.read_csv(path)


## Create shared skill mapping and ensure there are no unseen skills

In [ ]:
train_df = train_df.dropna(
    subset=["skill_name"]
).copy()

for q in test_dfs:
    test_dfs[q] = test_dfs[q].dropna(
        subset=["skill_name"]
    ).copy()

q_list = np.unique(
    train_df["skill_name"].values
)

q2idx = {
    q: idx
    for idx, q in enumerate(q_list)
}

num_q = len(q_list)

print("Number of skills:", num_q)

for q, df_q in test_dfs.items():

    unseen = set(
        df_q["skill_name"].unique()
    ) - set(q2idx.keys())

    print(
        f"{q}: unseen skills = {len(unseen)}"
    )

    if len(unseen) > 0:
        print(
            "WARNING: unseen skills:",
            list(unseen)[:10]
        )

Number of skills: 110
Q1: unseen skills = 0
Q2: unseen skills = 0
Q3: unseen skills = 0
Q4: unseen skills = 0


## Create custom dataset using shared mapping (This resembles the ASSIST2009 dataloader from the hcnoh repo)

In [ ]:
class ReliabilityASSIST2009(Dataset):

    def __init__(
        self,
        df,
        q2idx,
        seq_len=100
    ):
        super().__init__()

        self.dataset_dir = None

        self.q2idx = q2idx
        self.q_list = np.array(
            list(q2idx.keys())
        )

        self.num_q = len(q2idx)

        df = df.copy()

        df = df.dropna(
            subset=["skill_name"]
        )

        df = df.drop_duplicates(
            subset=["order_id", "skill_name"]
        )

        df = df.sort_values(
            by=["order_id"]
        )

        self.u_list = np.unique(
            df["user_id"].values
        )

        self.q_seqs = []
        self.r_seqs = []

        for u in self.u_list:

            df_u = df[
                df["user_id"] == u
            ]

            q_seq = np.array([
                q2idx[q]
                for q in df_u["skill_name"]
            ])

            r_seq = df_u["correct"].values.astype(
                np.float32
            )

            self.q_seqs.append(q_seq)
            self.r_seqs.append(r_seq)

        if seq_len:
            self.q_seqs, self.r_seqs = match_seq_len(
                self.q_seqs,
                self.r_seqs,
                seq_len
            )

        self.len = len(self.q_seqs)

    def __getitem__(self, index):
        return (
            self.q_seqs[index],
            self.r_seqs[index]
        )

    def __len__(self):
        return self.len

## Create training and testing datasets using custom loader

In [ ]:
SEQ_LEN = 100

train_dataset = ReliabilityASSIST2009(
    train_df,
    q2idx,
    seq_len=SEQ_LEN
)

print("Training sequences:", len(train_dataset))
print("Number of skills:", train_dataset.num_q)

test_datasets = {}

for q, df_q in test_dfs.items():

    test_datasets[q] = ReliabilityASSIST2009(
        df_q,
        q2idx,
        seq_len=SEQ_LEN
    )

    print(
        q,
        "sequences:",
        len(test_datasets[q])
    )

Training sequences: 5024
Number of skills: 110
Q1 sequences: 243
Q2 sequences: 300
Q3 sequences: 366
Q4 sequences: 271


## Create validation set to see AUCs and select best one for testing

In [ ]:
from torch.utils.data import random_split

# Make PyTorch dataset operations use CPU by default
torch.set_default_device("cpu")

VAL_RATIO = 0.10

train_size = int(
    len(train_dataset) * (1 - VAL_RATIO)
)

val_size = len(train_dataset) - train_size

generator = torch.Generator(device="cpu")
generator.manual_seed(SEED)

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=generator
)

print("Training sequences:", len(train_subset))
print("Validation sequences:", len(val_subset))

Training sequences: 4521
Validation sequences: 503


## Create Dataloaders

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_subset,
    batch_size=len(val_subset),
    shuffle=False,
    collate_fn=collate_fn
)

test_loaders = {}

for q, dataset in test_datasets.items():

    test_loaders[q] = DataLoader(
        dataset,
        batch_size=len(dataset),
        shuffle=False,
        collate_fn=collate_fn
    )

print("DataLoaders created.")

DataLoaders created.


## Create the model

In [ ]:
LEARNING_RATE = 0.0001
EMBED_DIM = 125
NUM_HEADS = 5
DROPOUT = 0.2
NUM_EPOCHS = 100

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = SAKT(
    num_q=num_q,
    n=SEQ_LEN,
    d=EMBED_DIM,
    num_attn_heads=NUM_HEADS,
    dropout=DROPOUT
).to(device)

optimizer = Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Device:", device)
print("Model:", model)

Device: cuda
Model: SAKT(
  (M): Embedding(220, 125)
  (E): Embedding(110, 125)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=125, out_features=125, bias=True)
  )
  (attn_dropout): Dropout(p=0.2, inplace=False)
  (attn_layer_norm): LayerNorm((125,), eps=1e-05, elementwise_affine=True)
  (FFN): Sequential(
    (0): Linear(in_features=125, out_features=125, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=125, out_features=125, bias=True)
    (4): Dropout(p=0.2, inplace=False)
  )
  (FFN_layer_norm): LayerNorm((125,), eps=1e-05, elementwise_affine=True)
  (pred): Linear(in_features=125, out_features=1, bias=True)
)


## Training function

In [ ]:
def evaluate_model(model, loader, device):

    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for data in loader:

            q, r, qshft, rshft, m = data

            q = q.long().to(device)
            r = r.long().to(device)
            qshft = qshft.long().to(device)
            rshft = rshft.to(device)
            m = m.to(device)

            p, _ = model(
                q,
                r,
                qshft
            )

            p = torch.masked_select(
                p,
                m
            )

            t = torch.masked_select(
                rshft,
                m
            )

            all_predictions.append(
                p.detach().cpu().numpy()
            )

            all_targets.append(
                t.detach().cpu().numpy()
            )

    predictions = np.concatenate(
        all_predictions
    )

    targets = np.concatenate(
        all_targets
    )

    auc = roc_auc_score(
        targets,
        predictions
    )

    return auc

def train_sakt(
    model,
    train_loader,
    val_loader,
    optimizer,
    num_epochs,
    device,
    checkpoint_path
):

    best_auc = -np.inf
    best_epoch = None

    auc_history = []
    loss_history = []

    for epoch in range(1, num_epochs + 1):

        model.train()

        epoch_losses = []

        for data in train_loader:

            q, r, qshft, rshft, m = data

            q = q.long().to(device)
            r = r.long().to(device)
            qshft = qshft.long().to(device)
            rshft = rshft.to(device)
            m = m.to(device)

            p, _ = model(
                q,
                r,
                qshft
            )

            p = torch.masked_select(
                p,
                m
            )

            t = torch.masked_select(
                rshft,
                m
            )

            loss = nn.functional.binary_cross_entropy(
                p,
                t
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(
                loss.detach().cpu().item()
            )

        mean_loss = np.mean(
            epoch_losses
        )

        val_auc = evaluate_model(
            model,
            val_loader,
            device
        )

        auc_history.append(val_auc)
        loss_history.append(mean_loss)

        print(
            f"Epoch {epoch:03d} | "
            f"AUC: {val_auc:.6f} | "
            f"Loss: {mean_loss:.6f}"
        )

        if val_auc > best_auc:

            best_auc = val_auc
            best_epoch = epoch

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"  → New best model "
                f"(AUC={best_auc:.6f})"
            )

    return (
        auc_history,
        loss_history,
        best_auc,
        best_epoch
    )

## Train the model

In [ ]:
CHECKPOINT_DIR = os.path.join(
    RELIABILITY_DIR,
    "sakt_reliability"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "sakt_best.pt"
)

print(CHECKPOINT_PATH)

auc_history, loss_history, best_val_auc, best_epoch = train_sakt(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=device,
    checkpoint_path=CHECKPOINT_PATH
)

print("\nTraining complete.")
print("Best validation AUC:", best_val_auc)
print("Best epoch:", best_epoch)

/content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/sakt_reliability/sakt_best.pt


/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))


Epoch 001 | AUC: 0.603003 | Loss: 0.653842
  → New best model (AUC=0.603003)
Epoch 002 | AUC: 0.671795 | Loss: 0.617322
  → New best model (AUC=0.671795)
Epoch 003 | AUC: 0.706681 | Loss: 0.595486
  → New best model (AUC=0.706681)
Epoch 004 | AUC: 0.726352 | Loss: 0.575666
  → New best model (AUC=0.726352)
Epoch 005 | AUC: 0.739250 | Loss: 0.562821
  → New best model (AUC=0.739250)
Epoch 006 | AUC: 0.749086 | Loss: 0.552062
  → New best model (AUC=0.749086)
Epoch 007 | AUC: 0.756183 | Loss: 0.545002
  → New best model (AUC=0.756183)
Epoch 008 | AUC: 0.761996 | Loss: 0.539892
  → New best model (AUC=0.761996)
Epoch 009 | AUC: 0.766537 | Loss: 0.535072
  → New best model (AUC=0.766537)
Epoch 010 | AUC: 0.770121 | Loss: 0.529991
  → New best model (AUC=0.770121)
Epoch 011 | AUC: 0.773352 | Loss: 0.527200
  → New best model (AUC=0.773352)
Epoch 012 | AUC: 0.776132 | Loss: 0.524193
  → New best model (AUC=0.776132)
Epoch 013 | AUC: 0.778419 | Loss: 0.523377
  → New best model (AUC=0.778419)

## Load the best checkpoint and test on 4 quartiles

In [ ]:
model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=device
    )
)

model.to(device)
model.eval()

print("Best checkpoint loaded.")

quartile_results = {}

for q in ["Q1", "Q2", "Q3", "Q4"]:

    auc = evaluate_model(
        model,
        test_loaders[q],
        device
    )

    quartile_results[q] = auc

    print(
        f"{q} AUC: {auc:.6f}"
    )

Best checkpoint loaded.
Q1 AUC: 0.808080
Q2 AUC: 0.743699
Q3 AUC: 0.747745
Q4 AUC: 0.757854


## Create a results table

In [ ]:
results_df = pd.DataFrame({
    "Ability Quartile": list(
        quartile_results.keys()
    ),
    "SAKT AUC": list(
        quartile_results.values()
    )
})

results_df

,Ability Quartile,SAKT AUC
0,Q1,0.808080
1,Q2,0.743699
2,Q3,0.747745
3,Q4,0.757854


## Save the results

In [ ]:
RESULTS_PATH = os.path.join(
    CHECKPOINT_DIR,
    "sakt_quartile_auc.csv"
)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "Saved results to:",
    RESULTS_PATH
)

Saved results to: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/sakt_reliability/sakt_quartile_auc.csv


## SAKT Reliability Experiment

The original HCNOH SAKT repository was not used directly for this experiment because its standard training pipeline automatically creates a single random train/test split and evaluates the model on that mixed test set. This would not allow us to evaluate the same trained model separately on each student-ability quartile. Instead, the SAKT implementation from the repository was retained, while the data loading, train/validation split, and evaluation procedures were handled in a separate notebook. This allowed the model to be trained once on 80% of students from each ability quartile and then tested independently on the held-out 20% of students in Q1, Q2, Q3, and Q4.

Using the previously selected SAKT configuration (learning rate = 0.0001, embedding dimension = 125, batch size = 128, sequence length = 100, 5 attention heads, and 0.2 dropout), the model achieved a best validation AUC of 0.8051 at epoch 98. On the held-out quartile test sets, SAKT achieved an AUC of 0.8081 for Q1, 0.7437 for Q2, 0.7477 for Q3, and 0.7579 for Q4. The results indicate that SAKT's predictive performance varied across ability groups, with the lowest-ability group achieving the highest AUC and the other three groups producing lower and relatively similar AUC values. These results provide initial evidence that model performance may depend on student ability, which will be investigated further using the other knowledge tracing models.
